<a href="https://colab.research.google.com/github/kiranascs25-dotcom/BIS_LAB/blob/main/Genetic_Algorithm_for_Routing_Problems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math

# Coordinates of depot and customers
coords = {
    1: (50, 50),
    2: (96, 24),
    3: (40, 5),
    4: (49, 8),
    5: (13, 7),
    6: (29, 89),
    7: (48, 30),
    8: (84, 39),
    9: (14, 47),
    10: (2, 24),
    11: (3, 82),
    12: (65, 10),
    13: (98, 52),
    14: (84, 25),
    15: (41, 69),
    16: (1, 65),
    17: (51, 71),
    18: (75, 83),
    19: (29, 32),
    20: (83, 3),
    21: (50, 93),
    22: (80, 94),
    23: (5, 42),
    24: (62, 70),
    25: (31, 62),
    26: (19, 97),
    27: (91, 75),
    28: (27, 49),
    29: (23, 15),
    30: (20, 70),
    31: (85, 60),
    32: (98, 85)
}

# Customer demands
demand = {
    1: 0, 2: 16, 3: 11, 4: 6, 5: 10,
    6: 7, 7: 12, 8: 16, 9: 6, 10: 16,
    11: 8, 12: 14, 13: 7, 14: 16, 15: 3,
    16: 22, 17: 18, 18: 19, 19: 1, 20: 14,
    21: 8, 22: 12, 23: 4, 24: 8, 25: 24,
    26: 24, 27: 2, 28: 10, 29: 15, 30: 2,
    31: 14, 32: 9
}

CAPACITY = 120
MAX_DISTANCE = 250

POPULATION_SIZE = 100
GENERATIONS = 1000
MUTATION_RATE = 0.1
CROSSOVER_RATE = 0.8


def distance(a, b):
    x1, y1 = coords[a]
    x2, y2 = coords[b]
    return math.sqrt((x1-x2)**2 + (y1-y2)**2)


def create_route(chromosome):
    routes = []
    route = [1]
    load = 0
    dist = 0
    current = 1

    for customer in chromosome:
        d = demand[customer]
        new_dist = dist + distance(current, customer) + distance(customer, 1)

        if load + d <= CAPACITY and new_dist <= MAX_DISTANCE:
            route.append(customer)
            load += d
            dist += distance(current, customer)
            current = customer
        else:
            route.append(1)
            routes.append(route)

            route = [1, customer]
            load = d
            dist = distance(1, customer)
            current = customer

    route.append(1)
    routes.append(route)

    return routes


def route_distance(route):
    total = 0

    for i in range(len(route) - 1):
        total += distance(route[i], route[i+1])

    return total


def fitness(chromosome):
    routes = create_route(chromosome)

    total_distance = sum(route_distance(r) for r in routes)

    # Small penalty for using more vehicles
    vehicle_cost = len(routes) * 30

    return total_distance + vehicle_cost


def create_population():
    customers = list(range(2, 33))
    population = []

    for _ in range(POPULATION_SIZE):
        chromosome = customers[:]
        random.shuffle(chromosome)
        population.append(chromosome)

    return population


def selection(population):
    tournament = random.sample(population, 5)
    return min(tournament, key=fitness)


def crossover(parent1, parent2):
    if random.random() > CROSSOVER_RATE:
        return parent1[:]

    start, end = sorted(
        random.sample(range(len(parent1)), 2)
    )

    child = [None] * len(parent1)

    child[start:end] = parent1[start:end]

    remaining = [
        x for x in parent2
        if x not in child
    ]

    index = 0

    for i in range(len(child)):
        if child[i] is None:
            child[i] = remaining[index]
            index += 1

    return child


def mutation(chromosome):
    if random.random() < MUTATION_RATE:
        a, b = random.sample(range(len(chromosome)), 2)
        chromosome[a], chromosome[b] = chromosome[b], chromosome[a]

    return chromosome


# Genetic Algorithm
population = create_population()

best_solution = None
best_cost = float("inf")

for generation in range(GENERATIONS):

    new_population = []

    for _ in range(POPULATION_SIZE):

        parent1 = selection(population)
        parent2 = selection(population)

        child = crossover(parent1, parent2)
        child = mutation(child)

        new_population.append(child)

    population = new_population

    current_best = min(population, key=fitness)
    current_cost = fitness(current_best)

    if current_cost < best_cost:
        best_cost = current_cost
        best_solution = current_best[:]

# Final result
routes = create_route(best_solution)

print("========== VEHICLE ROUTING PROBLEM ==========")
print("Genetic Algorithm")
print()

for i, route in enumerate(routes, 1):
    print("Vehicle", i, ":", " -> ".join(map(str, route)))
    print("Distance :", round(route_distance(route), 2))

print()
print("Number of Vehicles :", len(routes))
print("Total Distance     :", round(
    sum(route_distance(r) for r in routes), 2
))
print("Total Cost         :", round(best_cost, 2))

========== VEHICLE ROUTING PROBLEM ==========
Genetic Algorithm

Vehicle 1 : 1 -> 15 -> 17 -> 21 -> 26 -> 6 -> 11 -> 25 -> 1
Distance : 181.12
Vehicle 2 : 1 -> 29 -> 5 -> 10 -> 19 -> 28 -> 9 -> 23 -> 16 -> 30 -> 1
Distance : 225.03
Vehicle 3 : 1 -> 13 -> 20 -> 12 -> 4 -> 3 -> 7 -> 1
Distance : 190.56
Vehicle 4 : 1 -> 31 -> 8 -> 14 -> 2 -> 27 -> 32 -> 22 -> 18 -> 24 -> 1
Distance : 220.83

Number of Vehicles : 4
Total Distance     : 817.54
Total Cost         : 937.54
